# QAOA pour MaxCut

$$H_C = \sum_{(i,j) \in E} \frac12 (I - Z_i Z_j)$$
$$H_B = \sum_i X_i$$

État variationnel: $|\gamma,\beta\rangle = e^{-i\beta_p H_B} e^{-i\gamma_p H_C} \cdots e^{-i\beta_1 H_B} e^{-i\gamma_1 H_C} |+\rangle^{\otimes n}$

In [ ]:
import pennylane as qml
from pennylane import numpy as np
import matplotlib.pyplot as plt
import networkx as nx

### Problème MaxCut

MaxCut sur un graphe $G=(V,E)$:
$$\text{maximiser } C(x) = \sum_{(i,j) \in E} (1 - \delta_{x_i, x_j})$$

In [ ]:
edges = [(0, 1), (0, 2), (1, 2), (1, 3), (2, 3)]
G = nx.Graph()
G.add_edges_from(edges)
n_qubits = len(G.nodes)

pos = nx.spring_layout(G, seed=42)
nx.draw(G, pos, with_labels=True, node_color='lightblue', 
        node_size=500, font_size=16, font_weight='bold')
plt.title('Graphe MaxCut')
plt.show()

print(f'Sommets: {n_qubits}, Arêtes: {len(edges)}')

In [ ]:
dev = qml.device('default.qubit', wires=n_qubits, shots=None)

def qaoa_layer(gamma, beta, edges):
    for (u, v) in edges:
        qml.CNOT(wires=[u, v])
        qml.RZ(2 * gamma, wires=v)
        qml.CNOT(wires=[u, v])
    for i in range(n_qubits):
        qml.RX(2 * beta, wires=i)

In [ ]:
@qml.qnode(dev)
def qaoa_circuit(params, edges):
    p = params.shape[0]
    for i in range(n_qubits):
        qml.Hadamard(wires=i)
    for layer in range(p):
        qaoa_layer(params[layer, 0], params[layer, 1], edges)
    return qml.probs(wires=range(n_qubits))

In [ ]:
def maxcut_cost(params, edges):
    probs = qaoa_circuit(params, edges)
    cost = 0.0
    for (u, v) in edges:
        for bitstring, prob in enumerate(probs):
            if ((bitstring >> u) & 1) != ((bitstring >> v) & 1):
                cost += prob
    return -cost

In [ ]:
np.random.seed(42)
p_layers = 2
params = np.random.uniform(0, np.pi, (p_layers, 2), requires_grad=True)
print('Coût initial:', maxcut_cost(params, edges))

### Optimisation

Minimisation du coût $-\langle H_C \rangle$ via descente de gradient.

In [ ]:
opt = qml.AdamOptimizer(stepsize=0.1)
n_steps = 150
costs = []

for step in range(n_steps):
    params, c = opt.step_and_cost(maxcut_cost, params, edges)
    costs.append(c)
    if step % 20 == 0:
        print(f'Étape {step:3d}: C = {-c:.4f} (coût=-{c:.4f})')

In [ ]:
plt.figure(figsize=(8, 5))
plt.plot(-np.array(costs), linewidth=2)
plt.xlabel('Itération')
plt.ylabel('-⟨H_C⟩ (valeur à maximiser)')
plt.title('Convergence QAOA pour MaxCut')
plt.grid(True)
plt.show()

### Approximation ratio

$$\alpha = \frac{C_{\text{QAOA}}}{C_{\text{max}}}$$

Plus $\alpha$ est proche de 1, meilleure est l'approximation.

In [ ]:
def maxcut_bruteforce(edges):
    n = max(max(e) for e in edges) + 1
    best = 0
    for bits in range(2**n):
        cut = sum(1 for u, v in edges if ((bits >> u) & 1) != ((bits >> v) & 1))
        best = max(best, cut)
    return best

C_max = maxcut_bruteforce(edges)
C_qaoa = -np.min(costs)
alpha = C_qaoa / C_max
print(f'C_max (exact) = {C_max}')
print(f'C_QAOA = {C_qaoa:.4f}')
print(f'Approximation ratio α = {alpha:.4f}')

In [ ]:
# Distribution des coupes
probs = qaoa_circuit(params, edges)
cuts = []
for bitstring, prob in enumerate(probs):
    cut = sum(1 for u, v in edges if ((bitstring >> u) & 1) != ((bitstring >> v) & 1))
    cuts.append(cut)

plt.bar(range(2**n_qubits), cuts, width=0.6)
plt.xlabel('Bitstring')
plt.ylabel('Taille de la coupe')
plt.title('Coupes par bitstring')
plt.axhline(C_qaoa, color='r', linestyle='--', label=f'QAOA: {C_qaoa:.2f}')
plt.axhline(C_max, color='g', linestyle=':', label=f'Max: {C_max}')
plt.legend()
plt.grid(True, alpha=0.3)
plt.show()

## Questions

**Q1.** Augmenter le nombre de couches $p$ de 1 à 4 et tracer $\alpha(p)$. Comment l'approximation ratio évolue-t-il avec $p$?

**Q2.** Utiliser Qiskit `SPSA` optimizer au lieu de Adam. Comparer la convergence (nombre d'itérations pour atteindre $\alpha > 0.9$) et le nombre d'évaluations de la fonction de coût.

In [ ]:
# Q1: α en fonction de p
p_range = range(1, 5)
alphas = []

for p in p_range:
    pars = np.random.uniform(0, np.pi, (p, 2), requires_grad=True)
    opt_p = qml.AdamOptimizer(stepsize=0.1)
    for _ in range(100):
        pars, _ = opt_p.step_and_cost(maxcut_cost, pars, edges)
    c_final = -maxcut_cost(pars, edges)
    alphas.append(c_final / C_max)
    print(f'p={p}: C={c_final:.4f}, α={alphas[-1]:.4f}')

plt.plot(list(p_range), alphas, 'o-', linewidth=2)
plt.xlabel('Couches p')
plt.ylabel('α (Approximation ratio)')
plt.title('Qualité QAOA vs nombre de couches')
plt.grid(True)
plt.axhline(1.0, color='r', linestyle='--', alpha=0.5)
plt.show()